In [1]:
# ── 1. IMPORTS & REPRODUCIBILITY ───────────────────────────────
import os, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import RobertaTokenizerFast, RobertaModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize
import optuna
from optuna.samplers import TPESampler
from tqdm.auto import tqdm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE} | AMP Enabled")
os.makedirs('../results', exist_ok=True)
os.makedirs('../models', exist_ok=True)

Device: cuda | AMP Enabled


In [2]:
# ── 2. CONSTANTS, DATA & FOCAL LOSS ────────────────────────────
MODEL_NAME    = 'roberta-base'
MAX_LEN       = 128
BATCH_SIZE    = 16  # Optimized for 8GB VRAM
NUM_CLASSES   = 3
WLP_LAYERS    = [9, 10, 11, 12]

df_train = pd.read_csv('../data/train.csv')
df_val   = pd.read_csv('../data/val.csv')
df_test  = pd.read_csv('../data/test.csv')

with open('../data/class_weights.json') as f:
    cw_dict = {int(k): v for k, v in json.load(f).items()}
cw_tensor = torch.tensor([cw_dict[i] for i in range(3)], dtype=torch.float).to(DEVICE)

# FOCAL LOSS: Forces the model to focus on the 7.1% Neutral class
class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(weight=alpha, reduction='none')
    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        pt = torch.exp(-ce_loss)
        return (((1 - pt) ** self.gamma) * ce_loss).mean()

criterion = FocalLoss(alpha=cw_tensor, gamma=2.0)
tokeniser = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

class TransformerDataset(Dataset):
    def __init__(self, df, tokeniser, max_len):
        texts  = df['combined_text'].fillna('').astype(str).tolist()
        labels = df['label'].tolist()
        encoding = tokeniser(texts, max_length=max_len, padding='max_length', truncation=True, return_tensors='pt', return_token_type_ids=False)
        self.input_ids = encoding['input_ids']
        self.attention_mask = encoding['attention_mask']
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {'input_ids': self.input_ids[idx], 'attention_mask': self.attention_mask[idx], 'labels': self.labels[idx]}

train_loader = DataLoader(TransformerDataset(df_train, tokeniser, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TransformerDataset(df_val, tokeniser, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TransformerDataset(df_test, tokeniser, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)

In [3]:
# ── 3. FLATTENED CUSTOM ARCHITECTURE ───────────────────────────
class WeightedLayerPooling(nn.Module):
    def __init__(self, layer_indices):
        super().__init__()
        self.layer_indices = layer_indices
        self.layer_weights = nn.Parameter(torch.tensor([0.1, 0.1, 0.3, 0.5]))
    def forward(self, all_hidden_states):
        selected = torch.stack([all_hidden_states[i] for i in self.layer_indices], dim=0)
        weights = torch.softmax(self.layer_weights, dim=0).view(-1, 1, 1, 1)
        return (weights * selected).sum(dim=0)[:, 0, :]

class MultiSampleDropout(nn.Module):
    def __init__(self, p, n, in_features, out_features):
        super().__init__()
        self.dropouts = nn.ModuleList([nn.Dropout(p) for _ in range(n)])
        self.linear   = nn.Linear(in_features, out_features)
    def forward(self, x):
        if self.training:
            return torch.stack([self.linear(drop(x)) for drop in self.dropouts], dim=0).mean(dim=0)
        return self.linear(self.dropouts[0](x))

class RoBERTaOptimizedHead(nn.Module):
    def __init__(self, roberta_model, layer_indices, msd_n, msd_p, num_classes):
        super().__init__()
        self.roberta = roberta_model
        self.wlp = WeightedLayerPooling(layer_indices)
        
        # THE FIX: Flattened head. No deep 256->64 MLP. 
        # Keeps native 768 dim, uses Tanh (smoother than GELU), feeds straight to MSD.
        self.dense = nn.Linear(768, 768)
        self.activation = nn.Tanh()
        self.msd = MultiSampleDropout(p=msd_p, n=msd_n, in_features=768, out_features=num_classes)
        
    def forward(self, input_ids, attention_mask):
        out = self.roberta(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        return self.msd(self.activation(self.dense(self.wlp(out.hidden_states))))

In [4]:
# ── 4. OPTUNA TPE TUNING & LLRD ────────────────────────────────
scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

def get_llrd_optimizer(model, base_lr, decay_factor, weight_decay):
    param_groups = []
    # Custom head gets a 3x multiplier (NOT 10x, to prevent divergence)
    head_lr = base_lr * 3.0  
    
    param_groups.append({'params': model.msd.parameters(), 'lr': head_lr, 'weight_decay': weight_decay})
    param_groups.append({'params': model.dense.parameters(), 'lr': head_lr, 'weight_decay': weight_decay})
    param_groups.append({'params': model.wlp.parameters(), 'lr': head_lr, 'weight_decay': weight_decay})
    
    for i in range(11, -1, -1):
        layer_lr = base_lr * (decay_factor ** (11 - i))
        param_groups.append({'params': model.roberta.encoder.layer[i].parameters(), 'lr': layer_lr, 'weight_decay': weight_decay})
    
    param_groups.append({'params': model.roberta.embeddings.parameters(), 'lr': base_lr * (decay_factor ** 12), 'weight_decay': weight_decay})
    return AdamW(param_groups)

def objective(trial):
    base_lr = trial.suggest_float("base_lr", 8e-6, 3e-5, log=True)
    decay_factor = trial.suggest_float("decay_factor", 0.85, 0.95)
    msd_p = trial.suggest_float("msd_p", 0.2, 0.4)
    
    model = RoBERTaOptimizedHead(RobertaModel.from_pretrained(MODEL_NAME), WLP_LAYERS, 5, msd_p, NUM_CLASSES).to(DEVICE)
    optimizer = get_llrd_optimizer(model, base_lr, decay_factor, weight_decay=0.01)
    
    TRIAL_EPOCHS = 3 
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * len(train_loader) * TRIAL_EPOCHS), num_training_steps=len(train_loader) * TRIAL_EPOCHS)
    
    best_f1 = 0.0
    for epoch in range(1, TRIAL_EPOCHS + 1):
        model.train()
        for batch in tqdm(train_loader, desc=f"Trial {trial.number} | Epoch {epoch}/{TRIAL_EPOCHS}", leave=False):
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                logits = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
                loss = criterion(logits, batch['labels'].to(DEVICE))
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                logits = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
                all_preds.extend(logits.argmax(1).cpu().numpy())
                all_labels.extend(batch['labels'].cpu().numpy())
                
        vl_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        best_f1 = max(best_f1, vl_f1)
        trial.report(vl_f1, epoch)
        if trial.should_prune(): raise optuna.exceptions.TrialPruned()
            
    return best_f1

print("=" * 60)
print("STARTING OPTUNA WITH TPE SAMPLER & LLRD")
print("=" * 60)
study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=RANDOM_STATE), pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=10) # 10 trials is optimal without FGM overhead
best_params = study.best_params
print(f"Best Params: {best_params}")

[I 2026-04-26 21:29:08,172] A new study created in memory with name: no-name-093ee7d1-eea3-4fdc-bbd2-8bdaa2dc5693


STARTING OPTUNA WITH TPE SAMPLER & LLRD


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trial 0 | Epoch 1/3:   0%|          | 0/500 [00:00<?, ?it/s]

Trial 0 | Epoch 2/3:   0%|          | 0/500 [00:00<?, ?it/s]

...

Trial 8 | Epoch 1/3:   0%|          | 0/500 [00:00<?, ?it/s]

[I 2026-04-26 22:11:03,560] Trial 8 pruned. 
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trial 9 | Epoch 1/3:   0%|          | 0/500 [00:00<?, ?it/s]

[I 2026-04-26 22:12:07,352] Trial 9 pruned. 


Best Params: {'base_lr': 2.4040200829582646e-05, 'decay_factor': 0.8712339110678275, 'msd_p': 0.23636499344142015}


In [8]:
# ── 5. FINAL TRAINING LOOP ─────────────────────────────────────
print("\n" + "=" * 60)
print("FINAL TRAINING WITH WINNING PARAMETERS")
print("=" * 60)

FINAL_EPOCHS = 20
PATIENCE = 4

final_model = RoBERTaOptimizedHead(RobertaModel.from_pretrained(MODEL_NAME), WLP_LAYERS, 5, best_params['msd_p'], NUM_CLASSES).to(DEVICE)
optimizer = get_llrd_optimizer(final_model, best_params['base_lr'], best_params['decay_factor'], weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * len(train_loader) * FINAL_EPOCHS), num_training_steps=len(train_loader) * FINAL_EPOCHS)

best_val_f1 = -1.0
patience_ctr = 0
val_probas_for_calibration = []
val_labels_for_calibration = []

for epoch in range(1, FINAL_EPOCHS + 1):
    final_model.train()
    for batch in tqdm(train_loader, desc=f"Final Train Epoch {epoch}/{FINAL_EPOCHS}"):
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = final_model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
            loss = criterion(logits, batch['labels'].to(DEVICE))
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(final_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
    final_model.eval()
    all_preds, all_labels, all_proba = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            logits = final_model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(batch['labels'].cpu().numpy())
            all_proba.extend(torch.softmax(logits, 1).cpu().numpy())
            
    vl_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    print(f"  Epoch {epoch} | Val Macro F1: {vl_f1:.4f}")
    
    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        torch.save(final_model.state_dict(), '../models/roberta_optimized_best.pt')
        patience_ctr = 0
        val_probas_for_calibration = all_proba
        val_labels_for_calibration = all_labels
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE: 
            print("Early stopping triggered.")
            break


FINAL TRAINING WITH WINNING PARAMETERS


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Final Train Epoch 1/20:   0%|          | 0/500 [00:00<?, ?it/s]

  Epoch 1 | Val Macro F1: 0.6277


Final Train Epoch 2/20:   0%|          | 0/500 [00:00<?, ?it/s]

...

  Epoch 10 | Val Macro F1: 0.7115


Final Train Epoch 11/20:   0%|          | 0/500 [00:00<?, ?it/s]

  Epoch 11 | Val Macro F1: 0.7342


Final Train Epoch 12/20:   0%|          | 0/500 [00:00<?, ?it/s]

  Epoch 12 | Val Macro F1: 0.7326
Early stopping triggered.


In [9]:
# ── 6. THRESHOLD CALIBRATION & TEST SET EVAL ───────────────────
print("\n" + "=" * 60)
print("OPTIMIZING DECISION BOUNDARIES (THRESHOLD TUNING)")
print("=" * 60)

def threshold_objective(weights):
    weighted_proba = np.array(val_probas_for_calibration) * weights
    preds = np.argmax(weighted_proba, axis=1)
    return -f1_score(val_labels_for_calibration, preds, average='macro', zero_division=0)

bounds = [(0.1, 5.0), (0.1, 5.0), (0.1, 5.0)]
res = minimize(threshold_objective, [1.0, 1.0, 1.0], method='Nelder-Mead', bounds=bounds)
optimal_weights = res.x

print(f"Optimal Class Multipliers: Neg: {optimal_weights[0]:.2f} | Neu: {optimal_weights[1]:.2f} | Pos: {optimal_weights[2]:.2f}")

test_model = RoBERTaOptimizedHead(RobertaModel.from_pretrained(MODEL_NAME), WLP_LAYERS, 5, best_params['msd_p'], NUM_CLASSES).to(DEVICE)
test_model.load_state_dict(torch.load('../models/roberta_optimized_best.pt', map_location=DEVICE))
test_model.eval()

test_preds, test_labels, test_proba = [], [], []
with torch.no_grad():
    for batch in test_loader:
        logits = test_model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        test_labels.extend(batch['labels'].cpu().numpy())
        test_proba.extend(torch.softmax(logits, 1).cpu().numpy())

# Apply the calibrated weights to the test set
weighted_test_proba = np.array(test_proba) * optimal_weights
test_preds = np.argmax(weighted_test_proba, axis=1)

test_macro_f1 = f1_score(test_labels, test_preds, average='macro', zero_division=0)
test_acc = accuracy_score(test_labels, test_preds)
test_auc = roc_auc_score(label_binarize(test_labels, classes=[0,1,2]), np.array(test_proba), multi_class='ovr', average='macro')

print("\n" + "=" * 60)
print(f"FINAL SOTA EVALUATION | MACRO F1: {test_macro_f1:.4f}")
print("=" * 60)
print(f"  Accuracy  : {test_acc:.4f}")
print(f"  ROC-AUC   : {test_auc:.4f}")

# Save Results
layer_weights = torch.softmax(test_model.wlp.layer_weights, dim=0).detach().cpu().numpy()
results = {
    'model': 'RoBERTa Flattened Head (Optuna + LLRD + Calibrated)',
    'macro_f1': round(float(test_macro_f1), 4),
    'accuracy': round(float(test_acc), 4),
    'roc_auc': round(float(test_auc), 4),
    'best_params': best_params,
    'layer_weights': {f'layer_{l}': round(float(w), 4) for l, w in zip(WLP_LAYERS, layer_weights)}
}
with open('../results/06_results_roberta_optimized.json', 'w') as f:
    json.dump(results, f, indent=2)


OPTIMIZING DECISION BOUNDARIES (THRESHOLD TUNING)
Optimal Class Multipliers: Neg: 1.00 | Neu: 1.00 | Pos: 1.00


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



FINAL SOTA EVALUATION | MACRO F1: 0.7389
  Accuracy  : 0.8990
  ROC-AUC   : 0.8755
